# Hybrid Production Architecture

**Level:** Advanced · **Time:** 90 min

In this notebook, we simulate the components of a Hybrid architecture, demonstrating why "everything is an agent" is a dangerous anti-pattern.

We will cover 3 patterns:
1. **Deterministic Classification:** Routing to strict workflows vs dynamic agents.
2. **The Unnecessary Team Anti-Pattern:** Comparing the latency of a single agent vs a 3-agent team.
3. **The Policy Gateway:** Blocking an agent hallucination with deterministic code.

---
## Pattern 1: Deterministic Classification

We use a fast intent classifier to route the request. If the request is linear (password reset), we use a Workflow. If ambiguous (system down), we use an Agent.

In [ ]:
def linear_workflow():
    print("  -> [Workflow] Executing strict State Machine: Ask Email -> Verify OTP -> Reset.")

def diagnostic_agent():
    print("  -> [Agent] Executing ReAct loop: Query metrics, form hypothesis, repeat.")

def deterministic_router(user_input):
    print(f"\n[Router] Analyzing intent for: '{user_input}'")
    # Simulated intent classification
    if "password" in user_input.lower():
        print("[Router] Intent is LINEAR. Routing to Workflow.")
        linear_workflow()
    else:
        print("[Router] Intent is AMBIGUOUS. Routing to Agent.")
        diagnostic_agent()

deterministic_router("I forgot my password.")
deterministic_router("Why is the database slow?")


---
## Pattern 2: The Unnecessary Team Anti-Pattern

Using a multi-agent team for a simple task incurs massive latency and token taxes. We simulate the difference.

In [ ]:
def single_agent_task():
    start = time.time()
    print("\n[Single Agent] Querying DB...")
    time.sleep(0.5)  # Simulate DB call
    print("[Single Agent] Done. User is Premium.")
    return time.time() - start

def multi_agent_team_task():
    start = time.time()
    print("\n[Manager Agent] Delegating to DB Agent...")
    time.sleep(0.5)
    print("[DB Agent] Querying DB... Done. Returning to Manager...")
    time.sleep(0.5)
    print("[Manager Agent] Delegating to Formatting Agent...")
    time.sleep(0.5)
    print("[Formatting Agent] Formatting response... Done. Returning to Manager...")
    time.sleep(0.5)
    print("[Manager Agent] Done. User is Premium.")
    return time.time() - start

print(f"Single Agent Latency: {single_agent_task():.2f} seconds")
print(f"Multi-Agent Team Latency: {multi_agent_team_task():.2f} seconds")


---
## Pattern 3: The Policy Gateway

The LLM never gets the final say. A deterministic python wrapper checks the output against policy (e.g., regex for credit cards) before sending it.

In [ ]:
import re

def policy_gateway(agent_output):
    print(f"\n[Gateway] Intercepting agent output for validation...")
    
    # Simple regex to catch a simulated credit card number (16 digits)
    cc_regex = re.compile(r'\b\d{16}\b')
    
    if cc_regex.search(agent_output):
        print("🚨 [Gateway] BLOCK: PII detected in output. Agent hallucinated a CC number.")
        return "ERROR: Policy Violation"
    else:
        print("✅ [Gateway] PASS: Output is safe.")
        return agent_output

# Scenario A: Safe output
safe_output = "The user's subscription expires on Tuesday."
print(policy_gateway(safe_output))

# Scenario B: Agent hallucinates sensitive data
hallucinated_output = "I found the billing info. The card is 1234567812345678."
print(policy_gateway(hallucinated_output))
